In [25]:
import numpy as np

data = [["I love computer science", 1],
        ["I hate economics", 0],
        ["I love music", 1],
        ["I hate indoor games", 0],
        ["I like reading", 1],
        ["I hate rot learning", 0]]

# Tokenize
tokens = []
labels = []
for sentence, label in data:
    tokens.append(sentence.lower().split())
    labels.append(label)

print("tokens", tokens)
print("labels", labels)

tokens [['i', 'love', 'computer', 'science'], ['i', 'hate', 'economics'], ['i', 'love', 'music'], ['i', 'hate', 'indoor', 'games'], ['i', 'like', 'reading'], ['i', 'hate', 'rot', 'learning']]
labels [1, 0, 1, 0, 1, 0]


In [26]:
# Vocabulary
vocab_set = set()
for sentence in tokens:
    for word in sentence:
        vocab_set.add(word)
vocabulary = sorted(vocab_set)

vocabulary

['computer',
 'economics',
 'games',
 'hate',
 'i',
 'indoor',
 'learning',
 'like',
 'love',
 'music',
 'reading',
 'rot',
 'science']

In [27]:
word_to_id = {}

for i in range(len(vocabulary)):
    word = vocabulary[i]
    word_to_id[word] = i

print(word_to_id)

{'computer': 0, 'economics': 1, 'games': 2, 'hate': 3, 'i': 4, 'indoor': 5, 'learning': 6, 'like': 7, 'love': 8, 'music': 9, 'reading': 10, 'rot': 11, 'science': 12}


In [28]:
id_to_word = {}

for word in word_to_id:
    word_id = word_to_id[word]
    id_to_word[word_id] = word

print(id_to_word)

{0: 'computer', 1: 'economics', 2: 'games', 3: 'hate', 4: 'i', 5: 'indoor', 6: 'learning', 7: 'like', 8: 'love', 9: 'music', 10: 'reading', 11: 'rot', 12: 'science'}


In [29]:
window = 1
pairs = []

for sentence in tokens:
    ids = [word_to_id[w] for w in sentence]

    for i, center in enumerate(ids):
        for j in range(max(0, i - window),
                       min(len(ids), i + window + 1)):
            if i != j:
                pairs.append((center, ids[j]))

pairs

[(4, 8),
 (8, 4),
 (8, 0),
 (0, 8),
 (0, 12),
 (12, 0),
 (4, 3),
 (3, 4),
 (3, 1),
 (1, 3),
 (4, 8),
 (8, 4),
 (8, 9),
 (9, 8),
 (4, 3),
 (3, 4),
 (3, 5),
 (5, 3),
 (5, 2),
 (2, 5),
 (4, 7),
 (7, 4),
 (7, 10),
 (10, 7),
 (4, 3),
 (3, 4),
 (3, 11),
 (11, 3),
 (11, 6),
 (6, 11)]

In [30]:
print("Pairs:", len(pairs))

Pairs: 30


In [31]:
from pprint import pprint

# One hot encoding
embedding_lookup = {}

for word_id in id_to_word:
    vector = []
    for i in range(len(vocabulary)):
        if i == word_id:
            vector.append(1)
        else:
            vector.append(0)
    embedding_lookup[word_id] = vector

pprint(embedding_lookup)

{0: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 1: [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 2: [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 3: [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 4: [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
 5: [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
 6: [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
 7: [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
 8: [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
 9: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
 10: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
 11: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
 12: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]}


In [32]:
example_word = "love"
example_id = word_to_id[example_word]
print("[word] = ", example_word, "\n[id] = ", example_id, "\n[one hot] = ", embedding_lookup[example_id])

[word] =  love 
[id] =  8 
[one hot] =  [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]


In [33]:
# Choose one pair and look up the input vector
center_id, context_id = pairs[0]
input_vector = embedding_lookup[center_id]

print('Center word:', id_to_word[center_id])
print('Target context word:', id_to_word[context_id])
print('Input vector:', input_vector)

Center word: i
Target context word: love
Input vector: [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]


In [34]:
# One row per input word, one column per possible context word
import random

random.seed(0)
context_weights = []

for input_index in range(len(vocabulary)):
    weight_row = []
    for context_index in range(len(vocabulary)):
        weight = random.gauss(0, 1) * 0.01
        weight_row.append(weight)
    context_weights.append(weight_row)

print('Weight matrix shape:', (len(context_weights), len(context_weights[0])))

Weight matrix shape: (13, 13)


In [35]:
# Calculate each context word's score
context_scores = []

for context_index in range(len(vocabulary)):
    score = 0
    for input_index in range(len(vocabulary)):
        score += input_vector[input_index] * context_weights[input_index][context_index]
    context_scores.append(score)

for word_id in range(len(vocabulary)):
    print(id_to_word[word_id], context_scores[word_id])

computer 0.007685090969541892
economics 0.004252991158576464
games -0.023211499734925593
hate -0.0011590422906128793
i 0.009801990211840203
indoor 0.008011650401167723
learning -0.0033943540831298876
like -0.012126242150471978
love 0.0049129354930742795
music -0.011458279458387148
reading 0.013246786624104563
rot -0.0030626754659669343
science -0.009248190034741935


In [36]:
# Softmax: subtract the largest score before exponentiating for stability
import math

largest_score = context_scores[0]
for score in context_scores:
    if score > largest_score:
        largest_score = score

exp_scores = []
exp_total = 0
for score in context_scores:
    exp_score = math.exp(score - largest_score)
    exp_scores.append(exp_score)
    exp_total += exp_score

context_probabilities = []
for exp_score in exp_scores:
    context_probabilities.append(exp_score / exp_total)

for word_id in range(len(vocabulary)):
    print(id_to_word[word_id], context_probabilities[word_id])

predicted_id = 0
probability_sum = 0
for word_id in range(len(vocabulary)):
    probability = context_probabilities[word_id]
    probability_sum += probability
    if probability > context_probabilities[predicted_id]:
        predicted_id = word_id

print('Probability sum:', probability_sum)
print('Predicted context word:', id_to_word[predicted_id])
print('Target context word:', id_to_word[context_id])
print('Target probability:', context_probabilities[context_id])

computer 0.07760647839589173
economics 0.07734058176894752
games 0.07524536578523366
hate 0.07692314256979134
i 0.07777093750146806
indoor 0.0776318256618363
learning 0.07675138739678569
like 0.0760841203632386
love 0.07739163909335563
music 0.07613495869426533
reading 0.07803930451743432
rot 0.07677684841302672
science 0.07630340983872501
Probability sum: 1.0
Predicted context word: reading
Target context word: love
Target probability: 0.07739163909335563
